# Starting and Stopping Simulations

Author: Sam Eure

May 28, 2025

In [ ]:
# Imports (run once)

from air_sdk import AirApi, SimState
from air_sdk.endpoints import Simulation

In [ ]:
# Authentication (run once)
api = AirApi.with_ngc_config()
# OR api = AirApi.with_api_key(api_key="...")
# OR api = AirApi.with_device_login(email="...", org_num="...")
#    ^ use in terminal only — not supported in Jupyter notebooks

### Create a new simulation

In [ ]:
json_data = {
    'format': 'JSON',
    'name': 'JSON Sim',
    'content': {
        'nodes': {
            'node1': {
                'cpu': 2,
                'memory': 1024,
                'storage': 10,
                'os': 'generic/ubuntu2204',
                'cpu_arch': 'x86',
            },
        },
        'oob': False,
    },
}

sim: Simulation = api.simulations.import_from_data(**json_data)
sim.wait_for_state(SimState.INACTIVE, error_states=SimState.INVALID)

### Start the simulation
Request that the simulation be started. If successful, the simulation will transition from `INACTIVE` → `REQUESTING` → `PROVISIONING` → `PREPARE_BOOT` → `BOOTING` → `ACTIVE`.

In [ ]:
sim.start()
sim.wait_for_state(SimState.ACTIVE, error_states=SimState.INACTIVE)

sim.state

'ACTIVE'

### Shut down the simulation

Request that the simulation be shut down. Unless you specifically pass `create_checkpoint=False`, Air will default to saving a new checkpoint while shutting down. If successful, the simulation will transition from `ACTIVE` → `PREPARE_SHUTDOWN` → `SHUTTING_DOWN` → `SAVING` → `INACTIVE`.

In [ ]:
sim.shutdown()
sim.wait_for_state(SimState.INACTIVE)

for history in list(sim.get_history()):
    print(history.description)

Transitioning from `SAVING` to `INACTIVE` state.
Checkpoint `Checkpoint 2026-03-11 18:36:14` has been successfully saved.
Transitioning from `SHUTTING_DOWN` to `SAVING` state.
Simulation `JSON Sim` with ID `c0b5e717-9bd1-46dd-84c3-fdd323b32cb2` created by importing a topology in `JSON` format. Topology is scheduled for validation.
Initiating topology validation.
Validation complete. Initiating creation of simulation objects.
Topology successfully imported.
Transitioning from `IMPORTING` to `INACTIVE` state.
Transitioning from `INACTIVE` to `REQUESTING` state.
Transitioning from `REQUESTING` to `PROVISIONING` state.
Transitioning from `PROVISIONING` to `PREPARE_BOOT` state.
Transitioning from `PREPARE_BOOT` to `BOOTING` state.
Transitioning from `BOOTING` to `ACTIVE` state.
Transitioning from `ACTIVE` to `PREPARE_SHUTDOWN` state.
Transitioning from `PREPARE_SHUTDOWN` to `SHUTTING_DOWN` state.
Transitioning from `SHUTTING_DOWN` to `INACTIVE` state.

### Shut down the simulation without a checkpoint

Pass `create_checkpoint=False` when you request shutdown to not create a new checkpoint. The simulation will then transition from `ACTIVE` -> `PREPARE_PURGE` -> `PURGING` -> `INACTIVE`. 

In [ ]:
sim.shutdown(create_checkpoint=False)
sim.wait_for_state(SimState.INACTIVE)

for history in list(sim.get_history()):
    print(history.description)

Transitioning from `PURGING` to `INACTIVE` state.
Transitioning from `PREPARE_PURGE` to `PURGING` state.
Transitioning from `ACTIVE` to `PREPARE_PURGE` state.
Purge shutdown requested; the simulation will be purged and no checkpoint will be created.
Transitioning from `BOOTING` to `ACTIVE` state.
The simulation has successfully booted. Starting billing. Putting the simulation in an `ACTIVE` state.
Transitioning from `PREPARE_BOOT` to `BOOTING` state.
Transitioning from `PROVISIONING` to `PREPARE_BOOT` state.
Transitioning from `REQUESTING` to `PROVISIONING` state.
Transitioning from `INACTIVE` to `REQUESTING` state.
Boot requested without a checkpoint.
Transitioning from `IMPORTING` to `INACTIVE` state.
Topology successfully imported.
Validation complete. Initiating creation of simulation objects.
Initiating topology validation.
Simulation `JSON Sim` with ID `c0b5e717-9bd1-46dd-84c3-fdd323b32cb2` created by importing a topology in `JSON` format. Topology is scheduled for validation.


In [15]:
sim.state

'INACTIVE'

## Checkpoints <a id="checkpoints"></a>

When a shutdown creates a checkpoint, Air persists the simulation state so you can restore later. Use `sim.checkpoints` to list, update metadata, rebuild the simulation from a snapshot, or delete a checkpoint you no longer need.


### List checkpoints

Iterate `sim.checkpoints.list()`. You can optionally filter, e.g. `state='COMPLETE'`.


In [27]:
for cp in sim.checkpoints.list():
    print(f'{cp.name} (id={cp.id}) — state: {cp.state}')

Checkpoint 2025-09-24 18:44:55 (id=1d92e7b4-5f3a-4c81-9a2d-7b30f84c129e) — state: COMPLETE
Checkpoint 2025-09-24 18:44:55 (id=6af3c018-9b42-4f75-8d6e-1c9a3e507bd4) — state: DELETED
Checkpoint Renamed (id=e3b7a520-2149-4a6c-b8f3-0d57c9a41e86) — state: COMPLETE


You can optionally filter, e.g. `state='COMPLETE'`.

In [23]:
complete = list(sim.checkpoints.list(state='COMPLETE'))
print(f'Complete checkpoints: {len(complete)}')

Complete checkpoints: 2


### Update a checkpoint

Rename a checkpoint or toggle `favorite` with `update()` on the checkpoint object.

`favorite` tells Air to prefer keeping this checkpoint if it ever has to automatically delete older ones to stay within your org’s checkpoint limit.



In [ ]:
cp = sim.checkpoints.get('checkpoint-id')
cp.update(name='Updated snapshot', favorite=True)
print(f'Checkpoint id: {cp.id}')
print(f'Name: {cp.name}')
print(f'Favorite: {cp.favorite}')

Checkpoint id: 1d92e7b4-5f3a-4c81-9a2d-7b30f84c129e
Name: Updated snapshot
Favorite: True


### Rebuild the simulation

With the simulation ACTIVE, `rebuild()` tears down and recreate its nodes. Wait until `ACTIVE` again so the rebuild finishes.

In [ ]:
if sim.state != 'ACTIVE':
    sim.start()
    sim.wait_for_state(SimState.ACTIVE, error_states=SimState.INACTIVE)

sim.rebuild()
sim.wait_for_state(SimState.ACTIVE)
print(f'Rebuild finished; simulation state: {sim.state}')

Rebuild finished; simulation state: ACTIVE


### Rebuild from a specific checkpoint

If you pass a checkpoint into `rebuild()`, Air rebuilds the nodes and restores them from that checkpoint’s saved state.

In [ ]:
if sim.state != 'ACTIVE':
    sim.start()
    sim.wait_for_state(SimState.ACTIVE, error_states=SimState.INACTIVE)

cp = sim.checkpoints.get('checkpoint-id')  # pick a checkpoint; adjust as needed
sim.rebuild(checkpoint=cp.id)
sim.wait_for_state(SimState.ACTIVE)
print(f'Rebuilt from checkpoint: {cp.name} (id={cp.id})')
print(f'Simulation state: {sim.state}')

Rebuilt from checkpoint: Checkpoint 2026-04-29 18:31:03 (id=6af3c018-9b42-4f75-8d6e-1c9a3e507bd4)
Simulation state: ACTIVE


### Delete a checkpoint

In [ ]:
cp_to_remove = sim.checkpoints.get('checkpoint-id')
cp_to_remove.delete()